<div style="
background: linear-gradient(135deg, #f8f9fa 0%, #edf6f9 45%, #e8eaf6 100%);
padding: 40px;
border-radius: 20px;
text-align: center;
font-family: 'Segoe UI', sans-serif;
box-shadow: 0 8px 24px rgba(0,0,0,0.08);
border: 1px solid #dce3ea;
">

  <h1 style="
  color: #5c6b8a;
  font-size: 2.2em;
  margin: 0 0 8px 0;
  letter-spacing: 1px;
  font-weight: 700;">
  🤖 CP020003 — Artificial Intelligence 2026
  </h1>

  <h2 style="
  color: #7b8fa1;
  font-size: 1.3em;
  margin: 0 0 16px 0;
  font-weight: 400;">
  Khon Kaen University
  </h2>

  <hr style="
  border: 1px solid #c9d6df;
  width: 60%;
  margin: 18px auto;">

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    👨‍🏫 <strong style="color:#6c7aa1;">Author:</strong>
    Teerapong Panboonyuen (P'Kao)
  </p>

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    📧 <strong style="color:#6c7aa1;">Contact:</strong>
    teerapong.pa@chula.ac.th
  </p>

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    🏫 <strong style="color:#6c7aa1;">Course:</strong>
    AI 2026 @ KKU
  </p>

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    📦 <strong style="color:#6c7aa1;">GitHub:</strong>
    <a href="https://github.com/kaopanboonyuen/CP020003_ArtificialIntelligence_2026s1"
       style="color:#5b8def; text-decoration:none;">
       CP020003_ArtificialIntelligence_2026s1
    </a>
  </p>

  <hr style="
  border: 1px solid #c9d6df;
  width: 60%;
  margin: 18px auto;">

  <p style="
color: #6c757d;
font-size: 0.95em;
margin: 4px 0;">
📚 Built with inspiration from the open-source AI community:
<strong style="color:#7286a0;">
Python · Pandas · NumPy · scikit-learn · PyTorch · Hugging Face · Kaggle
</strong>
</p>

  <p style="
  color: #8a97a6;
  font-size: 0.9em;
  margin-top: 12px;
  font-style: italic;">
  "This notebook is open to everyone — including those who cannot afford university.
  Knowledge is for all. 🌏"
  </p>

</div>

## Week 8 — Time Series Forecasting: From Statistics to Transformers
### CP020003 Artificial Intelligence 2026 — In-Class Notebook

Today we forecast **real stock index prices** — five different ways. We'll go from 60-year-old statistical methods (AR, ARIMA) to the same class of model behind ChatGPT-style forecasting (attention/Transformers), and end with one scoreboard that puts every model on the same test set so you can see, honestly, which ones actually work.

> 🕐 **Kept short on purpose.** This whole notebook runs end-to-end in well under 30 minutes on a free Colab GPU (`Runtime → Change runtime type → T4 GPU`). We go for **concepts you can see**, not exhaustive tuning.

**The golden rule of this entire notebook:** we forecast the **future** from the **past**. Every split, every feature, every evaluation below respects time order. If you ever see a model peeking at data from *after* the point it's forecasting, that model is cheating — and in real trading, cheating models lose real money.

**Roadmap:**

| Level | Family | Models |
|---|---|---|
| 1 | Foundations | What is a time series? EDA, train/test split |
| 2 | Classical statistics | AR, MA, ARMA, ARIMA, SARIMA, VAR, Exponential Smoothing / Holt-Winters |
| 3 | Time series → tabular ML | Lag features → Linear Regression, Random Forest, XGBoost, LightGBM, CatBoost |
| 4 | Deep learning | MLP, RNN, LSTM, GRU, 1D CNN |
| 5 | Attention / Transformers | Self-attention demo + tour of modern architectures |
| 🏆 | Scoreboard | Every model, one leaderboard |


## 0. Setup 🔧

Colab already ships `numpy`, `pandas`, `matplotlib`, `scikit-learn`, `statsmodels`, and `torch`. We only need to add the gradient-boosting libraries.

In [ ]:
# Run once per Colab session — about 20-30 seconds
!pip -q install xgboost lightgbm catboost statsmodels --upgrade

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import io, zipfile, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from sklearn.metrics import mean_absolute_error, mean_squared_error

SEED = # Write your lucky number here

np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cpu":
    print("⚠️  No GPU detected — Runtime → Change runtime type → T4 GPU (not required, but faster for Level 4-5).")

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True

## 1. What *is* Time Series Forecasting? 🧭

**Time series** — a sequence of values measured over successive, ordered points in time: $x_1, x_2, \dots, x_t$. Order matters. Shuffle a time series and it stops meaning anything.

**Time series forecasting** — using the history $x_1, \dots, x_t$ to predict future values $x_{t+1}, x_{t+2}, \dots$

**Why it isn't "just regression":**

| Ordinary tabular ML | Time series |
|---|---|
| Rows are (usually) independent | Each point depends on the ones before it |
| Random train/test split is fine | Train/test split **must** respect time order |
| Shuffling data is harmless | Shuffling destroys the signal |
| "Leakage" mostly means duplicate rows | Leakage means seeing **the future** — the cardinal sin |

**Four things every series can contain:**
- **Trend** — long-run drift up or down
- **Seasonality** — a pattern that repeats at a *fixed, known* period (every 7 days, every 12 months)
- **Cyclic behaviour** — repeats, but not at a fixed period (business cycles, bull/bear markets)
- **Noise / residual** — whatever is left after removing the above; ideally unpredictable

**Stationarity** — a series is (weakly) stationary if its mean, variance, and autocorrelation don't change over time. Most classical models (AR, MA, ARIMA...) assume stationarity — raw stock prices are almost never stationary (they trend), which is exactly why we'll difference them below.


## 2. The Dataset 💹

Global stock market indices, daily, from the course repo:

```
stock-exchange/
    indexData.csv        # raw daily OHLCV per index
    indexInfo.csv         # which index belongs to which country / exchange / currency
    indexProcessed.csv    # cleaned OHLCV + CloseUSD  <- we use this one
```

13 indices are available (NYA, IXIC, HSI, 000001.SS, N225, N100, 399001.SZ, GSPTSE, NSEI, GDAXI, KS11, SSMI, TWII, J203.JO). We'll teach on **IXIC (NASDAQ Composite)** — long history, no missing trading days, priced in USD already.

In [ ]:
DATA_NAME = # Write your dataset name here
DATA_URL = f"https://github.com/kaopanboonyuen/CP020003_ArtificialIntelligence_2026s1/raw/main/dataset/{DATA_NAME}.zip"

resp = requests.get(DATA_URL)
z = zipfile.ZipFile(io.BytesIO(resp.content))
z.extractall("stock-exchange")

# the zip contains a nested stock-exchange/ folder — find the CSVs wherever they landed
import glob
processed_path = glob.glob("stock-exchange/**/indexProcessed.csv", recursive=True)[0]
info_path = glob.glob("stock-exchange/**/indexInfo.csv", recursive=True)[0]

df_all = pd.read_csv(processed_path)
df_info = pd.read_csv(info_path)
df_all["Date"] = pd.to_datetime(df_all["Date"])

print(df_all.shape)
df_info

In [ ]:
# Pick one index to teach on: NASDAQ Composite (IXIC), most recent ~6.5 years of daily data
INDEX = # Write your code here
raw = df_all[df_all["Index"] == INDEX].sort_values("Date").reset_index(drop=True)
raw = raw[raw["Date"] >= "2015-01-01"].reset_index(drop=True)

print(f"{INDEX}: {len(raw)} trading days, {raw['Date'].min().date()} → {raw['Date'].max().date()}")
raw.tail()

### 2.1 Exploratory Data Analysis (EDA) 🔍

Always look at your series before modelling it — plots reveal trend, volatility regimes, and outliers that summary statistics hide.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=False)

axes[0].plot(raw["Date"], raw["Close"], color="#2563eb")
axes[0].set_title(f"{INDEX} — Daily Close, {raw['Date'].min().date()} to {raw['Date'].max().date()}")
axes[0].set_ylabel("Close price (USD)")

axes[1].bar(raw["Date"], raw["Volume"], color="#94a3b8", width=1.5)
axes[1].set_title(f"{INDEX} — Daily Volume")

plt.tight_layout()
plt.show()

print(raw["Close"].describe())

### 2.2 Decomposition — Trend, Seasonality, Residual

`seasonal_decompose` splits the series into the three ingredients from Section 1. Daily stock data has no strong calendar seasonality, but decomposition is a core diagnostic you should run on *any* new series.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

close = # Write your code here
close.index = pd.RangeIndex(len(close))  # avoid statsmodels date-frequency issues

decomp = seasonal_decompose(close, model="additive", period=252, extrapolate_trend="freq")  # ~252 trading days/year

fig, axes = plt.subplots(4, 1, figsize=(11, 8), sharex=True)
axes[0].plot(decomp.observed, color="#2563eb"); axes[0].set_title("Observed")
axes[1].plot(decomp.trend, color="#16a34a");   axes[1].set_title("Trend")
axes[2].plot(decomp.seasonal, color="#d97706");axes[2].set_title("Seasonal (period=252d)")
axes[3].plot(decomp.resid, color="#dc2626");   axes[3].set_title("Residual")
plt.tight_layout()
plt.show()

## 3. The Golden Rule — Train / Test Split 🚫🔮

**Never shuffle. Never let the model see the future.** We hold out the **last 60 trading days (~3 months)** as test data. Every model in this notebook — classical, ML, and deep learning — is trained on data strictly *before* the test window and evaluated on that same 60-day window, so all scores land in the one scoreboard at the end on equal footing.

In [ ]:
TEST_SIZE = 60  # trading days (~3 months) held out for every model in this notebook

series = # Write your code here  # RangeIndex'd Close price series, defined above
split_idx = len(series) - TEST_SIZE

train, test = series.iloc[:split_idx], series.iloc[split_idx:]
print(f"Train: {len(train)} days | Test: {len(test)} days (last {TEST_SIZE} trading days)")

plt.plot(train.index, train.values, label="Train", color="#2563eb")
plt.plot(test.index, test.values, label="Test (never seen during fitting)", color="#dc2626")
plt.axvline(split_idx, color="gray", linestyle="--", linewidth=1)
plt.title(f"{INDEX} Close — Train / Test Split")
plt.legend()
plt.show()

We'll also keep a running scoreboard as we go — one row per model, filled in as each section finishes.

In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def mape(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)

def score(name, y_true, y_pred, family):
    results[name] = {
        "Family": family,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "MAPE (%)": mape(y_true, y_pred),
    }
    print(f"{name:22s} | MAE {results[name]['MAE']:9.2f} | RMSE {results[name]['RMSE']:9.2f} | MAPE {results[name]['MAPE (%)']:6.2f}%")

results = {}  # filled in across the whole notebook, used for the final scoreboard

## 4. Stationarity & the ADF Test 📐

The **Augmented Dickey-Fuller (ADF) test** checks for a unit root (non-stationarity):

- **H₀ (null hypothesis):** the series is non-stationary
- If **p-value < 0.05** → reject H₀ → the series **is** (likely) stationary

Raw stock prices trend, so we expect a high p-value. **Differencing** ($x_t - x_{t-1}$) usually fixes it — that's the "I" (Integrated) in ARIMA.

In [ ]:
from statsmodels.tsa.stattools import adfuller

def adf_report(x, label):
    stat, pvalue, *_ = adfuller(x.dropna())
    verdict = "✅ stationary" if pvalue < 0.05 else "❌ non-stationary"
    print(f"{label:16s} | ADF stat = {stat:8.3f} | p-value = {pvalue:.4f} | {verdict}")

adf_report(train, "Raw Close")
adf_report(train.diff(), "1st difference")

## 5. ACF & PACF — Choosing the Order 🎯

- **ACF (Autocorrelation Function)** — correlation of the series with its own lagged values. Tails off slowly → look at MA order (q).
- **PACF (Partial ACF)** — correlation with a lag *after* removing the effect of shorter lags. Cuts off sharply → look at AR order (p).

Rule of thumb: PACF spike at lag *p*, then drops → suggests **AR(p)**. ACF spike at lag *q*, then drops → suggests **MA(q)**. We plot both on the *differenced* (now-stationary) series.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(train.diff().dropna(), lags=30, ax=axes[0])
axes[0].set_title("ACF (differenced Close)")
plot_pacf(train.diff().dropna(), lags=30, ax=axes[1], method="ywm")
axes[1].set_title("PACF (differenced Close)")
plt.tight_layout()
plt.show()

## 6. Classical Time Series Models 📊

| Model | Captures | statsmodels class |
|---|---|---|
| AR(p) | value depends on its own past values | `ARIMA(order=(p,0,0))` |
| MA(q) | value depends on past *forecast errors* | `ARIMA(order=(0,0,q))` |
| ARMA(p,q) | both of the above | `ARIMA(order=(p,0,q))` |
| ARIMA(p,d,q) | ARMA + differencing for trend | `ARIMA(order=(p,d,q))` |
| SARIMA | ARIMA + a seasonal cycle | `SARIMAX(order, seasonal_order)` |
| Holt-Winters | trend + (optional) seasonality via exponential smoothing | `ExponentialSmoothing` |
| VAR | several correlated series forecasting *each other* | `VAR` |

Every model below is fit on `train` only and produces one static 60-day-ahead forecast — the honest, hard version of the task (no peeking at intermediate true values along the way).

### 6.0 Baseline — Naive Forecast (the model to beat)

The simplest possible forecast: **tomorrow = today**. For a near-random-walk series like a stock index, this is a *shockingly* strong baseline — and that's the point. Efficient markets are close to unpredictable day-to-day, so if a fancy model can't beat naive, it isn't adding value.

In [ ]:
naive_pred = np.repeat(train.iloc[-1], TEST_SIZE)
score("Naive (persistence)", test, naive_pred, "Baseline")

plt.plot(test.index, test.values, label="Actual", color="black")
plt.plot(test.index, naive_pred, label="Naive forecast", color="#94a3b8", linestyle="--")
plt.title("Naive baseline vs. actual")
plt.legend(); plt.show()

### 6.1 AR — Autoregressive

In [ ]:
ar_fit = __import__("statsmodels.tsa.arima.model", fromlist=["ARIMA"]).ARIMA(train, order=(5, 0, 0)).fit()
ar_pred = # Write your code here
score("AR(5)", test, ar_pred, "Classical")

### 6.2 MA — Moving Average

Watch this one closely — a pure MA model has no memory of the *level* or trend of a non-stationary series, so on raw trending prices it collapses toward the historical mean. This is a deliberate teaching moment: **MA alone is the wrong tool for a trending series.**

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

ma_fit = # Write your code here
ma_pred = ma_fit.forecast(TEST_SIZE)
score("MA(5)", test, ma_pred, "Classical")

plt.plot(test.index, test.values, label="Actual", color="black")
plt.plot(test.index, ma_pred, label="MA(5) forecast", color="#dc2626")
plt.title("MA(5) on a trending series — it can't follow the trend")
plt.legend(); plt.show()

### 6.3 ARMA

In [ ]:
arma_fit = ARIMA(train, order=(5, 0, 5)).fit()
arma_pred = arma_fit.forecast(TEST_SIZE)
score("ARMA(5,5)", test, arma_pred, "Classical")

### 6.4 ARIMA — adds differencing for the trend

This is the one most textbooks lead with. `d=1` means the model works on the *differenced* series internally, then integrates back — so it can track a trend that plain ARMA cannot.

In [ ]:
arima_fit = # Write your code here
arima_pred = arima_fit.get_forecast(TEST_SIZE)
arima_mean = arima_pred.predicted_mean
arima_ci = arima_pred.conf_int(alpha=0.05)  # 95% forecast interval

score("ARIMA(5,1,0)", test, arima_mean, "Classical")

plt.plot(train.index[-120:], train.values[-120:], label="Train (last 120d)", color="#94a3b8")
plt.plot(test.index, test.values, label="Actual", color="black")
plt.plot(test.index, arima_mean, label="ARIMA forecast", color="#2563eb")
plt.fill_between(test.index, arima_ci.iloc[:, 0], arima_ci.iloc[:, 1], color="#2563eb", alpha=0.15, label="95% forecast interval")
plt.title("ARIMA(5,1,0) forecast with 95% interval")
plt.legend(); plt.show()

**Forecast intervals matter as much as the point forecast.** A forecast interval widens the further out you predict, because uncertainty compounds — notice the cone shape above. A model that reports only a point forecast is quietly hiding how unsure it is.

**Residual diagnostics** — for a well-specified model, residuals (actual − fitted, on the *training* data) should look like white noise: no leftover pattern, no autocorrelation, roughly normal. `plot_diagnostics()` checks all of this at once; the **Ljung-Box test** formally tests "are these residuals just noise?" (p > 0.05 → yes, good).

In [ ]:
arima_fit.plot_diagnostics(figsize=(11, 7))
plt.tight_layout()
plt.show()

from statsmodels.stats.diagnostic import acorr_ljungbox
lb = acorr_ljungbox(arima_fit.resid, lags=[10], return_df=True)
print(lb)
print("→ p-value > 0.05 means residuals look like white noise (good, model captured the structure).")

### 6.5 SARIMA — ARIMA + a seasonal cycle

`SARIMAX(order=(p,d,q), seasonal_order=(P,D,Q,s))` adds a second, seasonal ARIMA structure with period `s`. We use a short `s=5` (trading week) purely to demonstrate the seasonal terms — daily index prices don't have strong weekly seasonality, so don't expect a big accuracy jump here; the point is the *mechanism*.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

sarima_fit = SARIMAX(train, order=(1, 1, 1), seasonal_order=(1, 1, 1, 5),
                      enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
sarima_pred = sarima_fit.forecast(TEST_SIZE)
score("SARIMA(1,1,1)(1,1,1,5)", test, sarima_pred, "Classical")

### 6.6 Exponential Smoothing & Holt-Winters

- **Simple Exponential Smoothing (SES):** weighted average of the past, weights decay exponentially — good for a level with no trend.
- **Holt's method:** SES + a trend component.
- **Holt-Winters:** Holt + a seasonal component. The full package.

In [ ]:
from statsmodels.tsa.holtwinters import SimpleExpSmoothing, ExponentialSmoothing

ses_fit = SimpleExpSmoothing(train, initialization_method="estimated").fit()
score("SES", test, ses_fit.forecast(TEST_SIZE), "Classical")

holt_fit = ExponentialSmoothing(train, trend="add", initialization_method="estimated").fit()
score("Holt (trend)", test, holt_fit.forecast(TEST_SIZE), "Classical")

hw_fit = ExponentialSmoothing(train, trend="add", seasonal="add", seasonal_periods=5, initialization_method="estimated").fit()
hw_pred = hw_fit.forecast(TEST_SIZE)
score("Holt-Winters", test, hw_pred, "Classical")

plt.plot(test.index, test.values, label="Actual", color="black")
plt.plot(test.index, hw_pred, label="Holt-Winters forecast", color="#16a34a")
plt.title("Holt-Winters vs. actual")
plt.legend(); plt.show()

### 6.7 VAR — Vector Autoregression (when series influence each other)

Every model so far was **univariate** — one series, forecast from its own past. **VAR** forecasts *several* series jointly, where each one depends on lagged values of *all* of them. Two well-correlated US indices — **NASDAQ (IXIC)** and **NYSE Composite (NYA)** — make a natural pair. VAR needs stationary inputs, so we difference both first.

*(VAR forecasts differenced values on a different scale from the rest of the notebook, so we report it separately rather than folding it into the main scoreboard.)*

In [ ]:
from statsmodels.tsa.api import VAR

nya = df_all[(df_all["Index"] == "NYA") & (df_all["Date"] >= "2015-01-01")].sort_values("Date").reset_index(drop=True)

bivariate = pd.concat([
    raw["Close"].rename("IXIC"),
    nya["Close"].rename("NYA"),
], axis=1).dropna().reset_index(drop=True)

biv_diff = bivariate.diff().dropna().reset_index(drop=True)
var_train, var_test = biv_diff.iloc[:-TEST_SIZE], biv_diff.iloc[-TEST_SIZE:]

var_fit = # Write your code here
var_forecast = var_fit.forecast(var_train.values[-5:], steps=TEST_SIZE)
var_forecast = pd.DataFrame(var_forecast, columns=["IXIC", "NYA"])

print(f"VAR(5) on Δ(IXIC), Δ(NYA)  |  IXIC ΔMAE = {mean_absolute_error(var_test['IXIC'], var_forecast['IXIC']):.2f}  |  NYA ΔMAE = {mean_absolute_error(var_test['NYA'], var_forecast['NYA']):.2f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(var_test["IXIC"].values, label="Actual Δ", color="black")
axes[0].plot(var_forecast["IXIC"].values, label="VAR forecast Δ", color="#2563eb")
axes[0].set_title("VAR — Δ IXIC"); axes[0].legend()
axes[1].plot(var_test["NYA"].values, label="Actual Δ", color="black")
axes[1].plot(var_forecast["NYA"].values, label="VAR forecast Δ", color="#d97706")
axes[1].set_title("VAR — Δ NYA"); axes[1].legend()
plt.tight_layout(); plt.show()

## 7. Time Series → Machine Learning 🌲

Classical models have a fixed structure (AR order, MA order...). The ML approach instead **reshapes the problem into an ordinary supervised-learning table**:

```
Time Series
     ↓
Lag features (lag_1, lag_2, lag_5, lag_20, rolling stats, volume...)
     ↓
Tabular dataset  X = [features at time t]     y = price[t+1]
     ↓
Any regression model: Linear Regression, Random Forest, XGBoost, LightGBM, CatBoost
```

**Crucial detail:** every feature at row `t` only uses information available *up to and including* `t` (we `shift(1)` before rolling). No feature is allowed to peek at `t+1` — that would be direct future leakage.

In [ ]:
feat = # Write your code here

for lag in [1, 2, 5, 20]:
    feat[f"lag_{lag}"] = feat["Close"].shift(lag)

feat["rolling_mean_20"] = feat["Close"].shift(1).rolling(20).mean()
feat["rolling_std_20"] = feat["Close"].shift(1).rolling(20).std()
feat["target"] = feat["Close"].shift(-1)  # y = price[t+1]

feat = feat.dropna().reset_index(drop=True)

FEATURE_COLS = ["lag_1", "lag_2", "lag_5", "lag_20", "rolling_mean_20", "rolling_std_20", "Volume"]
ml_split = len(feat) - TEST_SIZE

X_train, X_test = feat[FEATURE_COLS].iloc[:ml_split], feat[FEATURE_COLS].iloc[ml_split:]
y_train, y_test = feat["target"].iloc[:ml_split], feat["target"].iloc[ml_split:]

print(f"{len(X_train)} train rows | {len(X_test)} test rows | {len(FEATURE_COLS)} features")
feat[["Date"] + FEATURE_COLS + ["target"]].tail()

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

lr = LinearRegression().fit(X_train, y_train)
score("Linear Regression", y_test, lr.predict(X_test), "ML (tabular)")

rf = RandomForestRegressor(n_estimators=300, max_depth=6, random_state=42).fit(X_train, y_train)
score("Random Forest", y_test, rf.predict(X_test), "ML (tabular)")

xgb_model = xgb.XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.05, random_state=42)
xgb_model.fit(X_train, y_train)
score("XGBoost", y_test, xgb_model.predict(X_test), "ML (tabular)")

lgb_model = lgb.LGBMRegressor(n_estimators=300, max_depth=4, learning_rate=0.05, random_state=42, verbose=-1)
lgb_model.fit(X_train, y_train)
score("LightGBM", y_test, lgb_model.predict(X_test), "ML (tabular)")

cb_model = CatBoostRegressor(n_estimators=300, max_depth=4, learning_rate=0.05, random_state=42, verbose=False)
cb_model.fit(X_train, y_train)
score("CatBoost", y_test, cb_model.predict(X_test), "ML (tabular)")

> 💡 **Try it yourself:** which feature matters most? Tree models expose `.feature_importances_` — plot it for the Random Forest and see whether `lag_1` (yesterday's price) dominates everything else. For a near-random-walk series, it usually does — another sign the market is close to efficient.

In [ ]:
importances = pd.Series(rf.feature_importances_, index=FEATURE_COLS).sort_values()
importances.plot(kind="barh", color="#2563eb", figsize=(7, 4))
plt.title("Random Forest — feature importance")
plt.tight_layout(); plt.show()

## 8. Deep Learning for Time Series 🧠

Instead of hand-crafted lag features, deep sequence models take a **raw sliding window** of the last `SEQ_LEN` values and learn the pattern themselves. We normalize using statistics from the *training* data only (again — no peeking at test statistics), build overlapping windows, and hold out the same last 60 targets as everywhere else.

We climb the architecture ladder from the lecture:
**MLP → RNN → LSTM → GRU → 1D CNN.**

In [ ]:
SEQ_LEN = # Write your code here

close_vals = raw["Close"].values.astype("float32")
dl_split = len(close_vals) - TEST_SIZE

mu, sigma = close_vals[:dl_split].mean(), close_vals[:dl_split].std()
close_norm = (close_vals - mu) / sigma

Xs, ys = [], []
for i in range(len(close_norm) - SEQ_LEN):
    Xs.append(close_norm[i:i + SEQ_LEN])
    ys.append(close_norm[i + SEQ_LEN])
Xs, ys = np.array(Xs, dtype="float32"), np.array(ys, dtype="float32")

X_tr, X_te = Xs[:-TEST_SIZE], Xs[-TEST_SIZE:]
y_tr, y_te = ys[:-TEST_SIZE], ys[-TEST_SIZE:]

X_tr_t = torch.tensor(X_tr).to(DEVICE)
y_tr_t = torch.tensor(y_tr).to(DEVICE)
X_te_t = torch.tensor(X_te).to(DEVICE)
y_te_true = y_te * sigma + mu  # back to price scale, for scoring

def train_and_score(model, name, seq_input=True, epochs=150, lr=1e-2):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    Xtr_in = X_tr_t.unsqueeze(-1) if seq_input else X_tr_t
    Xte_in = X_te_t.unsqueeze(-1) if seq_input else X_te_t

    model.train()
    for epoch in range(epochs):
        opt.zero_grad()
        pred = model(Xtr_in)
        loss = loss_fn(pred, y_tr_t)
        loss.backward()
        opt.step()

    model.eval()
    with torch.no_grad():
        pred_norm = model(Xte_in).cpu().numpy()
    pred_price = pred_norm * sigma + mu
    score(name, y_te_true, pred_price, "Deep Learning")
    return pred_price

print(f"{X_tr.shape[0]} train windows | {X_te.shape[0]} test windows | window length {SEQ_LEN}")

### 8.1 MLP — the baseline

A plain feed-forward network sees the whole window at once, as a flat vector — no notion of order beyond whichever input position each value sits in.

In [ ]:
class MLP(nn.Module):
    def __init__(self, seq_len):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(seq_len, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

torch.manual_seed(42)
mlp_pred = train_and_score(MLP(SEQ_LEN), "MLP", seq_input=False)

### 8.2 RNN — recurrence

A recurrent network processes the window **one step at a time**, carrying a hidden state forward — the first architecture here that actually models sequence order explicitly.

In [ ]:
class RecurrentNet(nn.Module):
    def __init__(self, cell="rnn", hidden=32):
        super().__init__()
        rnn_cls = {"rnn": nn.RNN, "gru": nn.GRU, "lstm": nn.LSTM}[cell]
        self.rnn = rnn_cls(input_size=1, hidden_size=hidden, batch_first=True)
        self.fc = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :]).squeeze(-1)

torch.manual_seed(42)
rnn_pred = train_and_score(RecurrentNet("rnn"), "RNN")

### 8.3 LSTM — long short-term memory

Plain RNNs struggle with long windows (vanishing gradients). LSTM adds gated memory cells so the network can choose what to remember and what to forget over longer horizons.

In [ ]:
torch.manual_seed(42)
lstm_pred = train_and_score(RecurrentNet("lstm"), "LSTM")

### 8.4 GRU — a simpler LSTM

GRU merges LSTM's gates into fewer parameters — often nearly as accurate, faster to train.

In [ ]:
torch.manual_seed(42)
gru_pred = train_and_score(RecurrentNet("gru"), "GRU")

### 8.5 1D CNN — convolution over time

Instead of recurrence, a 1D CNN slides small learned filters across the window, stacking layers to grow the receptive field — the same idea behind Temporal CNN / TCN architectures used in production forecasting systems.

In [ ]:
class CNN1D(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(32, 1)
    def forward(self, x):
        x = x.transpose(1, 2)  # (batch, 1, seq_len)
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = self.pool(x).squeeze(-1)
        return self.fc(x).squeeze(-1)

torch.manual_seed(42)
cnn_pred = train_and_score(CNN1D(), "1D CNN")

plt.plot(y_te_true, label="Actual", color="black")
for name, pred, c in [("MLP", mlp_pred, "#94a3b8"), ("LSTM", lstm_pred, "#2563eb"), ("1D CNN", cnn_pred, "#dc2626")]:
    plt.plot(pred, label=name, color=c, alpha=0.8)
plt.title("Deep learning forecasts vs. actual (test window)")
plt.legend(); plt.show()

## 9. Attention & Transformers — the Modern Era 🔭

RNN/LSTM/GRU process the window step by step — slow, and long-range dependencies still have to survive being passed through many steps. **Attention** lets every position look directly at every other position in one shot, weighted by relevance.

**Core ideas, in forecasting order:**
- **Attention** — for each query position, compute a weighted sum over all other positions, weights = how "relevant" each one is.
- **Self-attention** — the query, key, and value all come from the *same* sequence.
- **Multi-head attention** — run several attention "views" in parallel, each free to learn a different kind of relationship (short-term momentum, long-term level, etc.), then combine.
- **Positional encoding** — attention has no built-in sense of order, so position information is injected explicitly.
- **Encoder / decoder** and **causal attention** — a forecasting model must only attend to the *past*, never the future; causal (masked) attention enforces that at the architecture level, same spirit as our train/test rule in Section 3.

**Real forecasting architectures built on this idea** (know the *names and one-line idea*, not the internals, for today):

| Model | One-line idea |
|---|---|
| Temporal Fusion Transformer (TFT) | Attention + gating, built for interpretable multi-horizon forecasts |
| Informer | Sparse attention so Transformers scale to very long sequences |
| Autoformer | Replaces attention with auto-correlation, decomposes trend/seasonal internally |
| FEDformer | Mixes attention with frequency-domain (Fourier) processing |
| PatchTST | Splits the series into patches (like image patches) before attending |
| N-BEATS / N-HiTS | Pure feed-forward, stacked, no attention or recurrence — surprisingly strong |
| TimesFM, Chronos, TimeGPT, TiDE | "Foundation models" for time series — pretrained on massive multi-domain data, forecast new series zero-shot |

**Don't just memorize the model names** — the through-line is: *what relationship in the data is each architecture built to exploit?*

### A tiny self-attention demo (concept, not a full model)

Below is 20 lines of raw self-attention — no training, just the mechanism — applied to our last 20-day window, so you can *see* which past days the mechanism assigns weight to. A real forecasting Transformer stacks many of these blocks and trains the weight matrices; this cell just exposes the core operation.

In [ ]:
torch.manual_seed(0)

window = torch.tensor(close_norm[-SEQ_LEN:]).float().unsqueeze(-1)  # (seq_len, 1)
d_model = 8

W_q = torch.randn(1, d_model) * 0.5
W_k = torch.randn(1, d_model) * 0.5
W_v = torch.randn(1, d_model) * 0.5

Q, K, V = window @ W_q, window @ W_k, window @ W_v
attn_scores = (Q @ K.T) / np.sqrt(d_model)

# causal mask: day i may only attend to days <= i (never the future)
mask = torch.triu(torch.ones(SEQ_LEN, SEQ_LEN), diagonal=1).bool()
attn_scores = attn_scores.masked_fill(mask, float("-inf"))
attn_weights = torch.softmax(attn_scores, dim=-1)

plt.figure(figsize=(6, 5))
plt.imshow(attn_weights.detach().numpy(), cmap="Blues")
plt.colorbar(label="attention weight")
plt.xlabel("Key day (attended TO)"); plt.ylabel("Query day (attending FROM)")
plt.title("Causal self-attention weights (untrained, illustrative)")
plt.show()

print("Notice the upper-right triangle is zero — causal masking means")
print("day i never attends to a day after i. That's the architectural")
print("version of the 'no future leakage' rule from Section 3.")

## 🏆 10. Final Scoreboard

Every model above — classical, tabular ML, and deep learning — was scored on **the exact same 60-day test window** of `IXIC` Close prices. Lower is better on every metric. The **Naive baseline** is the line every other model has to beat to be worth its complexity.

> ⚠️ **One honest caveat before you read this table.** The classical models (Section 6) forecast the whole 60-day window **blind** from a single cut-off point — a genuinely hard task, since day-59 errors compound on day-1 errors. The ML and deep-learning models (Sections 7–8) are evaluated **one step at a time**, each prediction built from the *true* recent prices up to that point — an easier task. That's why ML/DL will tend to look stronger below: it isn't automatically "a better model," it's partly "an easier version of the question." Comparing forecasting methods fairly is itself a real skill — always ask *what information was each model actually allowed to use* before trusting a leaderboard.

In [ ]:
leaderboard = # Write your code here
leaderboard.index.name = "Model"
leaderboard = leaderboard.sort_values("RMSE")

def highlight_naive(row):
    return ["background-color: #fef9c3" if row.name.startswith("Naive") else "" for _ in row]

leaderboard.style.apply(highlight_naive, axis=1).format({"MAE": "{:.2f}", "RMSE": "{:.2f}", "MAPE (%)": "{:.2f}"})

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors = {"Baseline": "#94a3b8", "Classical": "#2563eb", "ML (tabular)": "#16a34a", "Deep Learning": "#dc2626"}
bar_colors = [colors[f] for f in leaderboard["Family"]]

ax.barh(leaderboard.index[::-1], leaderboard["RMSE"][::-1], color=bar_colors[::-1])
ax.set_xlabel("RMSE on the 60-day test window (lower is better)")
ax.set_title(f"{INDEX} — 60-Day Forecast RMSE, All Models")

from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=c, label=f) for f, c in colors.items()], loc="lower right")
plt.tight_layout()
plt.show()

best_model = leaderboard.index[0]
naive_rmse = leaderboard.loc[[i for i in leaderboard.index if i.startswith("Naive")][0], "RMSE"]
print(f"Best model: {best_model}  (RMSE {leaderboard.loc[best_model, 'RMSE']:.2f})")
print(f"Naive baseline RMSE: {naive_rmse:.2f}")
print(f"→ Beat naive by {(1 - leaderboard.loc[best_model, 'RMSE']/naive_rmse) * 100:.1f}%" if leaderboard.loc[best_model, "RMSE"] < naive_rmse else "→ Nothing beat naive here — a real and common result for daily stock prices.")

### How to actually *read* a forecast plot (not just admire it)

1. **Check it against the naive baseline first.** If a complex model can't beat "tomorrow = today," the complexity isn't earning its keep — this is *especially* common for daily stock prices, which are close to a random walk.
2. **Look at the interval, not just the line.** A point forecast with no uncertainty band is overclaiming confidence — prefer models (ARIMA, SARIMA) that give you one.
3. **Check the residuals**, not just the final-epoch loss — patterned residuals mean the model missed structure it could have captured.
4. **Never trust a backtest that could have leaked the future.** Re-check your train/test boundary and your feature engineering (`shift(1)` before rolling!) every time.
5. **MAPE lets you compare across price *levels***, MAE/RMSE are in the original units (USD here) — report the one that answers your actual question.


## Recap — What You Just Ran ✅

| Level | Family | Models you ran | Key idea |
|---|---|---|---|
| 1 | Foundations | — | Trend / seasonality / stationarity, chronological train/test split |
| 2 | Classical | Naive, AR, MA, ARMA, ARIMA, SARIMA, SES, Holt, Holt-Winters, VAR | Explicit statistical structure, ACF/PACF to choose orders, ADF for stationarity |
| 3 | Tabular ML | Linear Regression, Random Forest, XGBoost, LightGBM, CatBoost | Reshape the series into lag features, any regressor applies |
| 4 | Deep Learning | MLP, RNN, LSTM, GRU, 1D CNN | Learn the pattern directly from a raw sliding window |
| 5 | Attention | Causal self-attention demo + model tour | Every position attends to every past position at once |

**Exercises to try after class:**
- Swap `INDEX = "IXIC"` for another index from `indexInfo` (try `"HSI"` or `"GDAXI"`) and re-run everything — does the ranking change?
- Change `TEST_SIZE` to 20 or 120 — does the best model stay the best?
- Add a `day_of_week` feature to the ML section — does it help at all? (Hint: think about *why* it might not, for financial data.)
- Try `SEQ_LEN = 60` for the deep learning section — longer memory, but also fewer training windows. What happens?


---

<div style="
background: linear-gradient(135deg, #fafafa 0%, #eef6f9 50%, #e8eaf6 100%);
padding: 30px;
border-radius: 18px;
text-align: center;
font-family: 'Segoe UI', sans-serif;
box-shadow: 0 6px 18px rgba(0,0,0,0.06);
border: 1px solid #dce3ea;
">

  <h2 style="
  color: #5c6b8a;
  margin: 0 0 12px 0;
  font-size: 1.8em;
  font-weight: 700;">
  🎉 Well Done!
  </h2>

  <p style="
  color: #495057;
  font-size: 1.05em;
  margin: 6px 0;">
  You've completed the Week 9 Notebook for
  <strong style="color:#6c7aa1;">
  CP020003 — AI 2026 @ KKU
  </strong>
  </p>

  <!--
  <p style="
  color: #6c757d;
  font-size: 0.95em;
  margin-top: 12px;">
  Next week we dive into
  <strong style="color:#5b8def;">
  Supervised Learning
  </strong>
  — scikit-learn, train/test splits, and your first ML model 🚀
  </p>
  -->

  <hr style="
  border: 1px solid #c9d6df;
  width: 50%;
  margin: 16px auto;">

  <p style="
  color: #7d8790;
  font-size: 0.9em;
  font-style: italic;
  margin-bottom: 6px;">
  "Shared freely so that everyone, everywhere, can learn AI."
  </p>

  <p style="
  color: #8a97a6;
  font-size: 0.85em;">
  — Teerapong Panboonyuen (P'Kao) · teerapong.pa@chula.ac.th
  </p>

</div>